# Notebook 16 — AutoLaparo External Source-Counterpart Stress Test

This notebook reproduces only the AutoLaparo results used in the manuscript's
external stress-test table.

The reported protocol retains five native AutoLaparo phases and projects them
into four predefined operational source-counterpart groups:

```text
AutoLaparo phase 1  Preparation
    → Preparation_Access
    → {P1 TrocarPlacement, P2 Preparation}

AutoLaparo phase 2  DividingLigamentAndPeritoneum
    → Dissection_Exposure
    → {P3 CalotTriangleDissection, P5 GallbladderDissection}

AutoLaparo phase 3  DividingUterineVesselsAndLigament
    → Control_Cutting_Division
    → P4 ClippingCutting

AutoLaparo phase 4  TransectingTheVagina
    → Control_Cutting_Division
    → P4 ClippingCutting

AutoLaparo phase 5  SpecimenRemoval
    → Specimen_Handling
    → {P6 GallbladderPackaging, P8 GallbladderRetraction}
```

Native phase `6` (Suturing) and phase `7` (Washing) are excluded from the
reported protocol. No sensitivity-analysis variant is generated.

## Important reproduction rule

The reported AutoLaparo experiment groups sampled S-STKG rows by `g_ratio`
only. It does **not** split rows further by `p_ratio`. This dataset-specific
fragmentation rule is retained exactly because it produces the 794
uncertainty-selected queries reported in the manuscript.

For every retained phase-aligned fragment, the notebook:

- extracts motif signatures using the frozen source-trained motif extractor;
- selects one maximum-entropy motif in the original eight-phase source space;
- projects the Dirichlet-calibrated source-hub distribution into the four
  retained counterpart groups;
- evaluates the local target-support prototype baseline;
- evaluates hub-only consultation; and
- applies the support-calibrated reliability gate used in the manuscript.

The outer evaluation is leave-one-case-out. Prototype construction, local-scale
selection, global and counterpart-specific mixture-weight selection, and
reliability estimation use only the support cases of the current fold. Labels
from the held-out case are used only for final metric calculation.

The cache is generated automatically under `RESULTS_DIR`. The timestamped
result directory contains only the three manuscript-oriented files described
below.


## Generated results

Each run creates one timestamped directory containing exactly three files:

1. `AutoLaparo_ExternalStressTestSummary.csv`  
   Contains the Table 7 values: retained queries, target-support baseline
   accuracy, hub-only accuracy, final DMET-SSTKG accuracy, micro gain, and
   mean case-level gain.

2. `AutoLaparo_ExternalStressTestPredictions.csv`  
   Contains one held-out result per retained query, including local,
   hub-only, and final reliability-gated predictions. Every summary value can
   be recomputed from this file.

3. `AutoLaparo_ExternalStressTestConfig.json`  
   Records the fixed mapping, excluded phases, original `g_ratio`-only
   fragmentation, input artifacts, support-only parameter-selection protocol,
   communication accounting, and manuscript reference values.

The pipeline requires 10 cases and exactly 794 retained queries before saving
the three result files. The automatically generated cache is stored separately
and is not a manuscript output.


## Environment

Use the same environment as the source and other evaluation notebooks:

```bash
pip install torch==2.4.1 torch-geometric pandas numpy tqdm
```

Select the PyTorch build appropriate for the local CPU or CUDA environment.
The notebook uses CUDA when available and otherwise runs on CPU.


In [ ]:
from __future__ import annotations

import json
import random
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATv2Conv
from tqdm.auto import tqdm

In [ ]:
# =========================================================
# 1) PATH CONFIGURATION — EDIT THESE FOUR LOCAL PATHS
# =========================================================

from pathlib import Path

# Autolapora S-STKG data.
# The released repository counterpart is:
# S-STKG_Data/Autolapora_STKG
REMOTE_SSTKG_ROOT = Path(
"/path/AutoLaparo_STKG_ROLE"
)

# Motif-extractor checkpoint produced by Notebook 09.
# This legacy run stores the checkpoint as best_motif_model.pt.
MOTIF_CHECKPOINT = Path(
    "/path/best_motif_model.pt"
)

# Dirichlet-conformal hub artifact produced by Notebook 11.
HUB_FILE = Path(
    "/path/Shareable_Motif_Hub_DIRICHLET_CONFORMAL_PROTOTYPE.pt"
)

# Writable parent directory for the generated cache and timestamped results.
RESULTS_DIR = Path(
    "/..../Autolapora"
)

# Managed cache generated automatically from REMOTE_SSTKG_ROOT.
# The user does not edit this path.
REMOTE_CACHE_DIR = (
    RESULTS_DIR
    / "Autolapora_binary_cache"
)

# A missing cache is created automatically.
# An incompatible cache is rebuilt using the original g_ratio-only protocol.
FORCE_REPROCESS_CACHE = False

In [ ]:
# =========================================================
# 2) FIXED STUDY CONFIGURATION
# =========================================================

SEED = 42
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
EPS = 1e-12

CHUNK_SIZE = 100
MIN_CHUNK_FRAMES = 10

NUM_SOURCE_PHASES = 8
NODE_FEATURE_DIM = 11
EDGE_FEATURE_DIM = 3
HIDDEN_DIM = 128
MOTIF_DIM = 128
MOTIF_WINDOW = 16
MOTIF_STRIDE = 8

CASE_ID_COL = 7
G_RATIO_COL = 8
P_RATIO_COL = 9
VIRTUAL_FLAG_COL = 10

REMOVE_CASE_ID_FEATURE = True
REMOVE_G_RATIO_FEATURE = False
REMOVE_P_RATIO_FEATURE = False

EXPECTED_REMOTE_CASES = 10
EXPECTED_RETAINED_QUERIES = 794
CACHE_SCHEMA_VERSION = 1
CACHE_FRAME_GROUPING = "g_ratio_only"
FRAME_KEY_TOLERANCE = 1e-6

LOCAL_SCALE_GRID = [
    1.0,
    2.0,
    5.0,
    10.0,
]
ALPHA_GRID = [
    0.0,
    0.025,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.50,
    0.65,
    0.75,
]
RELIABILITY_PRIOR_STRENGTH = 5.0

QUERY_BYTES = 128 * 4
RESPONSE_BYTES = 8 * 2
BYTES_PER_CONSULTATION = (
    QUERY_BYTES
    + RESPONSE_BYTES
)

MANUSCRIPT_REFERENCE = {
    "retained_queries": 794,
    "target_support_baseline_accuracy_percent": 50.00,
    "hub_only_consultation_accuracy_percent": 54.53,
    "final_dmet_sstkg_accuracy_percent": 56.93,
    "micro_gain_vs_baseline_pp": 6.93,
    "mean_case_level_gain_vs_baseline_pp": 6.08,
}

SOURCE_PHASE_NAMES = {
    1: "TrocarPlacement",
    2: "Preparation",
    3: "CalotTriangleDissection",
    4: "ClippingCutting",
    5: "GallbladderDissection",
    6: "GallbladderPackaging",
    7: "CleaningCoagulation",
    8: "GallbladderRetraction",
}

AUTOLAPARO_PHASE_NAMES = {
    1: "Preparation",
    2: "DividingLigamentAndPeritoneum",
    3: "DividingUterineVesselsAndLigament",
    4: "TransectingTheVagina",
    5: "SpecimenRemoval",
    6: "Suturing",
    7: "Washing",
}

# Fixed final mapping used in the manuscript.
# source_model_indices are 0-based model/hub indices.
COUNTERPART_SPECS = {
    1: {
        "target_label_id": 0,
        "counterpart_name": "Preparation_Access",
        "source_original_phase_ids": [1, 2],
        "source_model_indices": [0, 1],
        "source_phase_names": [
            "TrocarPlacement",
            "Preparation",
        ],
    },
    2: {
        "target_label_id": 1,
        "counterpart_name": "Dissection_Exposure",
        "source_original_phase_ids": [3, 5],
        "source_model_indices": [2, 4],
        "source_phase_names": [
            "CalotTriangleDissection",
            "GallbladderDissection",
        ],
    },
    3: {
        "target_label_id": 2,
        "counterpart_name": "Control_Cutting_Division",
        "source_original_phase_ids": [4],
        "source_model_indices": [3],
        "source_phase_names": [
            "ClippingCutting",
        ],
    },
    4: {
        "target_label_id": 2,
        "counterpart_name": "Control_Cutting_Division",
        "source_original_phase_ids": [4],
        "source_model_indices": [3],
        "source_phase_names": [
            "ClippingCutting",
        ],
    },
    5: {
        "target_label_id": 3,
        "counterpart_name": "Specimen_Handling",
        "source_original_phase_ids": [6, 8],
        "source_model_indices": [5, 7],
        "source_phase_names": [
            "GallbladderPackaging",
            "GallbladderRetraction",
        ],
    },
}

TARGET_LABEL_TO_NAME = {
    int(spec["target_label_id"]): str(
        spec["counterpart_name"]
    )
    for spec in COUNTERPART_SPECS.values()
}

print(f"Device: {DEVICE}")
print(
    "Retained AutoLaparo phases:",
    sorted(COUNTERPART_SPECS),
)
print(
    "Excluded AutoLaparo phases:",
    [6, 7],
)


In [ ]:
# =========================================================
# 3) HELPERS
# =========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def paths_overlap(path_a, path_b):
    path_a = Path(
        path_a
    ).expanduser().resolve()
    path_b = Path(
        path_b
    ).expanduser().resolve()

    return (
        path_a == path_b
        or path_a in path_b.parents
        or path_b in path_a.parents
    )


def l2_normalize(values):
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    if values.size == 0:
        return values

    if values.ndim == 1:
        return values / (
            np.linalg.norm(values)
            + EPS
        )

    return values / (
        np.linalg.norm(
            values,
            axis=1,
            keepdims=True,
        )
        + EPS
    )


def softmax_array(values, axis=-1):
    values = np.asarray(
        values,
        dtype=np.float64,
    )
    values = (
        values
        - np.max(
            values,
            axis=axis,
            keepdims=True,
        )
    )
    exponentials = np.exp(values)

    return exponentials / (
        exponentials.sum(
            axis=axis,
            keepdims=True,
        )
        + EPS
    )


def normalized_entropy(probabilities):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )
    probabilities = np.clip(
        probabilities,
        EPS,
        1.0,
    )
    probabilities = probabilities / (
        probabilities.sum()
        + EPS
    )

    return float(
        -(
            probabilities
            * np.log(probabilities)
        ).sum()
        / max(
            np.log(len(probabilities)),
            EPS,
        )
    )


def read_single_fragment_label(label_path):
    label_path = Path(label_path)
    label_table = pd.read_csv(
        label_path,
        sep=None,
        engine="python",
    )
    label_table.columns = [
        str(column).strip()
        for column in label_table.columns
    ]

    if "phase_id" in label_table.columns:
        label_column = "phase_id"
    elif len(label_table.columns) >= 2:
        label_column = label_table.columns[1]
    else:
        raise ValueError(
            "A phase-label column could not be inferred "
            f"from {label_path}."
        )

    values = sorted(
        label_table[label_column]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if len(values) != 1:
        raise ValueError(
            "Each phase-aligned fragment must contain exactly "
            f"one phase label; found {values} in {label_path}."
        )

    return int(values[0])


def target_name(target_label_id):
    target_label_id = int(
        target_label_id
    )

    if target_label_id not in TARGET_LABEL_TO_NAME:
        raise ValueError(
            "Unknown counterpart target label: "
            f"{target_label_id}"
        )

    return TARGET_LABEL_TO_NAME[
        target_label_id
    ]


def logical_payload_mib(
    number_of_queries,
):
    return (
        int(number_of_queries)
        * BYTES_PER_CONSULTATION
        / (1024.0 ** 2)
    )


def validate_input_paths():
    remote_root = Path(
        REMOTE_SSTKG_ROOT
    ).expanduser().resolve()
    cache_root = Path(
        REMOTE_CACHE_DIR
    ).expanduser().resolve()
    motif_checkpoint = Path(
        MOTIF_CHECKPOINT
    ).expanduser().resolve()
    hub_file = Path(
        HUB_FILE
    ).expanduser().resolve()
    results_root = Path(
        RESULTS_DIR
    ).expanduser().resolve()

    if not remote_root.is_dir():
        raise FileNotFoundError(
            "AutoLaparo S-STKG directory was not found:\n"
            f"{remote_root}"
        )

    case_directories = sorted(
        path
        for path in remote_root.iterdir()
        if (
            path.is_dir()
            and path.name.startswith("Case_")
        )
    )

    if len(case_directories) != EXPECTED_REMOTE_CASES:
        raise RuntimeError(
            "The retained AutoLaparo cohort must contain "
            f"{EXPECTED_REMOTE_CASES} Case_* directories; "
            f"found {len(case_directories)}."
        )

    if paths_overlap(
        remote_root,
        cache_root,
    ):
        raise RuntimeError(
            "REMOTE_CACHE_DIR must be separate from "
            "REMOTE_SSTKG_ROOT."
        )

    if paths_overlap(
        remote_root,
        results_root,
    ):
        raise RuntimeError(
            "RESULTS_DIR must be separate from "
            "REMOTE_SSTKG_ROOT."
        )

    if not motif_checkpoint.is_file():
        raise FileNotFoundError(
            "Motif checkpoint was not found:\n"
            f"{motif_checkpoint}"
        )

    if not hub_file.is_file():
        raise FileNotFoundError(
            "Dirichlet-conformal hub was not found:\n"
            f"{hub_file}"
        )

    results_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"[OK] AutoLaparo S-STKG: {remote_root}")
    print(f"[OK] Retained cases      : {len(case_directories)}")
    print(f"[OK] Managed cache       : {cache_root}")
    print(f"[OK] Motif checkpoint    : {motif_checkpoint}")
    print(f"[OK] Hub artifact        : {hub_file}")
    print(f"[OK] Results root        : {results_root}")

    return case_directories


set_seed(SEED)

In [ ]:
# =========================================================
# 4) AUTOLAPARO S-STKG CACHE
# =========================================================

class AutoLaparoCounterpartDataset(Dataset):
    def __init__(
        self,
        root_dir,
        cache_dir,
        chunk_size=100,
        minimum_chunk_frames=10,
        force_reprocess=False,
    ):
        self.root_dir = Path(
            root_dir
        ).expanduser().resolve()
        self.cache_dir = Path(
            cache_dir
        ).expanduser().resolve()
        self.chunk_size = int(
            chunk_size
        )
        self.minimum_chunk_frames = int(
            minimum_chunk_frames
        )

        if paths_overlap(
            self.root_dir,
            self.cache_dir,
        ):
            raise RuntimeError(
                "The cache and source S-STKG directories "
                "must be separate."
            )

        super().__init__(
            str(self.root_dir)
        )

        cache_is_valid, reason = (
            self._cache_is_valid()
        )

        if force_reprocess:
            print(
                "[INFO] Existing AutoLaparo cache "
                "will be replaced intentionally."
            )
            self._remove_cache_safely()
            self._build_cache_atomically()

        elif cache_is_valid:
            print(
                "[OK] Existing manuscript-compatible "
                "AutoLaparo cache is reused: "
                f"{self.cache_dir}"
            )

        else:
            if self.cache_dir.exists():
                print(
                    "[INFO] Rebuilding incompatible cache. "
                    f"Reason: {reason}"
                )
                self._remove_cache_safely()

            self._build_cache_atomically()

        self.files = sorted(
            self.cache_dir.glob(
                "frag_*.pt"
            )
        )

        if len(self.files) != int(
            EXPECTED_RETAINED_QUERIES
        ):
            raise RuntimeError(
                "The final AutoLaparo cache must contain "
                f"{EXPECTED_RETAINED_QUERIES} retained "
                f"fragments; found {len(self.files)}."
            )

    @property
    def manifest_path(self):
        return (
            self.cache_dir
            / "cache_manifest.json"
        )

    def _remove_cache_safely(self):
        if not self.cache_dir.exists():
            return

        protected = {
            Path("/").resolve(),
            Path.home().resolve(),
            self.root_dir,
        }

        if (
            self.cache_dir in protected
            or paths_overlap(
                self.cache_dir,
                self.root_dir,
            )
        ):
            raise RuntimeError(
                "Unsafe cache deletion was blocked."
            )

        shutil.rmtree(
            self.cache_dir
        )

    def _inspect_cache_files(
        self,
        files,
    ):
        case_ids = set()
        raw_phase_ids = set()

        for file_path in files:
            try:
                cache_object = torch.load(
                    file_path,
                    map_location="cpu",
                    weights_only=False,
                )
            except Exception as error:
                return False, (
                    f"cache file cannot be read: "
                    f"{file_path}: {error}"
                )

            if (
                not isinstance(
                    cache_object,
                    dict,
                )
                or "graph_seq"
                not in cache_object
            ):
                return False, (
                    "unsupported cache object in "
                    f"{file_path}"
                )

            graph_sequence = cache_object[
                "graph_seq"
            ]

            if not graph_sequence:
                return False, (
                    "empty graph sequence in "
                    f"{file_path}"
                )

            metadata = dict(
                cache_object.get(
                    "metadata",
                    {},
                )
                or {}
            )
            case_id = str(
                metadata.get(
                    "case_id",
                    "",
                )
            )
            raw_phase_id = metadata.get(
                "raw_phase_id",
                None,
            )

            if not case_id:
                return False, (
                    "case_id metadata is absent in "
                    f"{file_path}"
                )

            if raw_phase_id is None:
                return False, (
                    "raw_phase_id metadata is absent in "
                    f"{file_path}"
                )

            raw_phase_id = int(
                raw_phase_id
            )

            if (
                raw_phase_id
                not in COUNTERPART_SPECS
            ):
                return False, (
                    "cache contains an excluded raw phase "
                    f"{raw_phase_id} in {file_path}"
                )

            expected_target = int(
                COUNTERPART_SPECS[
                    raw_phase_id
                ][
                    "target_label_id"
                ]
            )
            cached_target = int(
                metadata.get(
                    "target_label_id",
                    graph_sequence[0]
                    .y.item(),
                )
            )

            if cached_target != expected_target:
                return False, (
                    "cache target-label mapping differs "
                    f"in {file_path}"
                )

            case_ids.add(
                case_id
            )
            raw_phase_ids.add(
                raw_phase_id
            )

        if len(case_ids) != int(
            EXPECTED_REMOTE_CASES
        ):
            return False, (
                "cache case count differs from the "
                f"manuscript cohort: {len(case_ids)}"
            )

        if raw_phase_ids != set(
            COUNTERPART_SPECS
        ):
            return False, (
                "cache retained raw phases differ: "
                f"{sorted(raw_phase_ids)}"
            )

        return True, (
            "cache files match the manuscript cohort"
        )

    def _cache_is_valid(self):
        if not self.cache_dir.is_dir():
            return False, (
                "cache directory does not exist"
            )

        files = sorted(
            self.cache_dir.glob(
                "frag_*.pt"
            )
        )

        if len(files) != int(
            EXPECTED_RETAINED_QUERIES
        ):
            return False, (
                "cache fragment count is "
                f"{len(files)}, expected "
                f"{EXPECTED_RETAINED_QUERIES}"
            )

        (
            files_are_valid,
            files_reason,
        ) = self._inspect_cache_files(
            files
        )

        if not files_are_valid:
            return False, files_reason

        # A legacy cache generated by the original script has no manifest.
        # It is accepted only after full file/count/case/mapping validation.
        if not self.manifest_path.is_file():
            return True, (
                "validated legacy g_ratio-only cache"
            )

        try:
            manifest = json.loads(
                self.manifest_path.read_text(
                    encoding="utf-8"
                )
            )
        except Exception as error:
            return False, (
                f"cache manifest cannot be read: {error}"
            )

        required_manifest_values = {
            "cache_schema_version": (
                CACHE_SCHEMA_VERSION
            ),
            "source_root": str(
                self.root_dir
            ),
            "chunk_size": (
                self.chunk_size
            ),
            "minimum_chunk_frames": (
                self.minimum_chunk_frames
            ),
            "frame_grouping": (
                CACHE_FRAME_GROUPING
            ),
            "num_fragments": int(
                EXPECTED_RETAINED_QUERIES
            ),
            "raw_to_target_mapping": {
                str(raw_phase_id): int(
                    spec[
                        "target_label_id"
                    ]
                )
                for raw_phase_id, spec
                in COUNTERPART_SPECS.items()
            },
        }

        for (
            key,
            expected_value,
        ) in required_manifest_values.items():
            if manifest.get(key) != expected_value:
                return False, (
                    f"manifest field {key!r} differs"
                )

        return True, files_reason

    def _clean_node_features(
        self,
        frame_nodes,
    ):
        frame_nodes = np.asarray(
            frame_nodes,
            dtype=np.float32,
        ).copy()

        if REMOVE_CASE_ID_FEATURE:
            frame_nodes[
                :,
                CASE_ID_COL,
            ] = 0.0

        if REMOVE_G_RATIO_FEATURE:
            frame_nodes[
                :,
                G_RATIO_COL,
            ] = 0.0

        if REMOVE_P_RATIO_FEATURE:
            frame_nodes[
                :,
                P_RATIO_COL,
            ] = 0.0

        return frame_nodes

    @staticmethod
    def _g_ratio_mask(
        table,
        g_ratio,
    ):
        # This intentionally mirrors the original AutoLaparo script:
        # rows are grouped by g_ratio only, not by (g_ratio, p_ratio).
        return np.isclose(
            table.iloc[
                :,
                0,
            ].to_numpy(
                dtype=float
            ),
            float(g_ratio),
            rtol=0.0,
            atol=FRAME_KEY_TOLERANCE,
        )

    def _build_cache_atomically(self):
        temporary_dir = (
            self.cache_dir.parent
            / f".{self.cache_dir.name}.building"
        )

        if temporary_dir.exists():
            shutil.rmtree(
                temporary_dir
            )

        temporary_dir.mkdir(
            parents=True,
            exist_ok=False,
        )

        global_index = 0
        skipped_excluded = 0
        skipped_short = 0
        case_fragment_counts = {}

        try:
            case_directories = sorted(
                path
                for path in self.root_dir.iterdir()
                if (
                    path.is_dir()
                    and path.name.startswith(
                        "Case_"
                    )
                )
            )

            for case_directory in tqdm(
                case_directories,
                desc=(
                    "Building original g_ratio-only "
                    "AutoLaparo cache"
                ),
            ):
                case_fragment_counts[
                    case_directory.name
                ] = 0

                prefixes = sorted(
                    {
                        path.name.removesuffix(
                            "_nodes.txt"
                        )
                        for path in case_directory.glob(
                            "*_nodes.txt"
                        )
                    }
                )

                for prefix in prefixes:
                    paths = {
                        "nodes": (
                            case_directory
                            / f"{prefix}_nodes.txt"
                        ),
                        "edges": (
                            case_directory
                            / f"{prefix}_edges.txt"
                        ),
                        "edge_features": (
                            case_directory
                            / (
                                f"{prefix}"
                                "_edge_features.txt"
                            )
                        ),
                        "labels": (
                            case_directory
                            / f"{prefix}_labels.txt"
                        ),
                    }

                    missing = [
                        str(path)
                        for path in paths.values()
                        if not path.is_file()
                    ]

                    if missing:
                        raise FileNotFoundError(
                            "Incomplete S-STKG fragment "
                            f"{prefix}:\n"
                            + "\n".join(missing)
                        )

                    raw_phase_id = (
                        read_single_fragment_label(
                            paths["labels"]
                        )
                    )

                    if (
                        raw_phase_id
                        not in COUNTERPART_SPECS
                    ):
                        skipped_excluded += 1
                        continue

                    counterpart = (
                        COUNTERPART_SPECS[
                            raw_phase_id
                        ]
                    )
                    target_label_id = int(
                        counterpart[
                            "target_label_id"
                        ]
                    )

                    node_table = pd.read_csv(
                        paths["nodes"]
                    )
                    edge_table = pd.read_csv(
                        paths["edges"]
                    )
                    edge_feature_table = (
                        pd.read_csv(
                            paths[
                                "edge_features"
                            ]
                        )
                    )

                    # Exact original fragmentation rule.
                    unique_g_ratios = sorted(
                        node_table.iloc[
                            :,
                            G_RATIO_COL,
                        ]
                        .astype(float)
                        .unique()
                        .tolist()
                    )

                    for chunk_start in range(
                        0,
                        len(unique_g_ratios),
                        self.chunk_size,
                    ):
                        chunk_g_ratios = (
                            unique_g_ratios[
                                chunk_start:
                                chunk_start
                                + self.chunk_size
                            ]
                        )

                        if (
                            len(chunk_g_ratios)
                            < self.minimum_chunk_frames
                        ):
                            skipped_short += 1
                            continue

                        graph_sequence = []

                        for g_ratio in (
                            chunk_g_ratios
                        ):
                            node_mask = np.isclose(
                                node_table.iloc[
                                    :,
                                    G_RATIO_COL,
                                ].to_numpy(
                                    dtype=float
                                ),
                                float(g_ratio),
                                rtol=0.0,
                                atol=(
                                    FRAME_KEY_TOLERANCE
                                ),
                            )
                            frame_nodes = (
                                node_table.loc[
                                    node_mask
                                ].to_numpy(
                                    dtype=np.float32,
                                    copy=True,
                                )
                            )
                            frame_nodes = (
                                self._clean_node_features(
                                    frame_nodes
                                )
                            )

                            edge_mask = (
                                self._g_ratio_mask(
                                    edge_table,
                                    g_ratio,
                                )
                            )
                            edge_feature_mask = (
                                self._g_ratio_mask(
                                    edge_feature_table,
                                    g_ratio,
                                )
                            )
                            frame_edges = (
                                edge_table.loc[
                                    edge_mask
                                ].to_numpy()
                            )
                            frame_edge_features = (
                                edge_feature_table.loc[
                                    edge_feature_mask
                                ].to_numpy()
                            )

                            if (
                                len(frame_edges)
                                != len(
                                    frame_edge_features
                                )
                            ):
                                raise RuntimeError(
                                    "Edge and edge-feature "
                                    "rows are misaligned in "
                                    f"{prefix}."
                                )

                            if len(frame_edges):
                                edge_index = (
                                    torch.tensor(
                                        frame_edges[
                                            :,
                                            2:4,
                                        ].astype(
                                            np.int64
                                        ),
                                        dtype=torch.long,
                                    )
                                    .t()
                                    .contiguous()
                                )
                            else:
                                edge_index = (
                                    torch.zeros(
                                        (2, 0),
                                        dtype=torch.long,
                                    )
                                )

                            if len(
                                frame_edge_features
                            ):
                                edge_attr = torch.tensor(
                                    frame_edge_features[
                                        :,
                                        4:7,
                                    ].astype(
                                        np.float32
                                    ),
                                    dtype=torch.float,
                                )
                            else:
                                edge_attr = (
                                    torch.zeros(
                                        (0, 3),
                                        dtype=torch.float,
                                    )
                                )

                            graph = Data(
                                x=torch.tensor(
                                    frame_nodes,
                                    dtype=torch.float,
                                ),
                                edge_index=edge_index,
                                edge_attr=edge_attr,
                                y=torch.tensor(
                                    [target_label_id],
                                    dtype=torch.long,
                                ),
                            )
                            graph.is_v = (
                                graph.x[
                                    :,
                                    VIRTUAL_FLAG_COL,
                                ]
                                == 1
                            )

                            # Do not impose a one-virtual-node check here.
                            # The original g_ratio-only grouping may combine
                            # rows carrying different p_ratio values.
                            graph_sequence.append(
                                graph
                            )

                        cache_object = {
                            "graph_seq": (
                                graph_sequence
                            ),
                            "metadata": {
                                "case_id": (
                                    case_directory.name
                                ),
                                "prefix": prefix,
                                "raw_phase_id": int(
                                    raw_phase_id
                                ),
                                "raw_phase_name": (
                                    AUTOLAPARO_PHASE_NAMES[
                                        raw_phase_id
                                    ]
                                ),
                                "target_label_id": int(
                                    target_label_id
                                ),
                                "counterpart_name": str(
                                    counterpart[
                                        "counterpart_name"
                                    ]
                                ),
                                "chunk_start_gr": float(
                                    chunk_g_ratios[0]
                                ),
                                "chunk_end_gr": float(
                                    chunk_g_ratios[-1]
                                ),
                            },
                        }

                        torch.save(
                            cache_object,
                            temporary_dir
                            / (
                                f"frag_"
                                f"{global_index:06d}.pt"
                            ),
                        )
                        global_index += 1
                        case_fragment_counts[
                            case_directory.name
                        ] += 1

            if global_index != int(
                EXPECTED_RETAINED_QUERIES
            ):
                raise RuntimeError(
                    "The original g_ratio-only fragmentation "
                    "did not reproduce the manuscript cohort: "
                    f"generated {global_index} retained "
                    f"fragments, expected "
                    f"{EXPECTED_RETAINED_QUERIES}."
                )

            empty_cases = [
                case_id
                for case_id, count
                in case_fragment_counts.items()
                if count == 0
            ]

            if empty_cases:
                raise RuntimeError(
                    "No retained counterpart fragments "
                    "were created for:\n"
                    + "\n".join(empty_cases)
                )

            manifest = {
                "cache_schema_version": (
                    CACHE_SCHEMA_VERSION
                ),
                "source_root": str(
                    self.root_dir
                ),
                "chunk_size": (
                    self.chunk_size
                ),
                "minimum_chunk_frames": (
                    self.minimum_chunk_frames
                ),
                "frame_grouping": (
                    CACHE_FRAME_GROUPING
                ),
                "raw_to_target_mapping": {
                    str(raw_phase_id): int(
                        spec[
                            "target_label_id"
                        ]
                    )
                    for raw_phase_id, spec
                    in COUNTERPART_SPECS.items()
                },
                "num_fragments": int(
                    global_index
                ),
                "case_fragment_counts": (
                    case_fragment_counts
                ),
                "skipped_excluded_segments": int(
                    skipped_excluded
                ),
                "skipped_short_chunks": int(
                    skipped_short
                ),
            }

            (
                temporary_dir
                / "cache_manifest.json"
            ).write_text(
                json.dumps(
                    manifest,
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )

            if self.cache_dir.exists():
                self._remove_cache_safely()

            temporary_dir.rename(
                self.cache_dir
            )

        except Exception:
            if temporary_dir.exists():
                shutil.rmtree(
                    temporary_dir
                )
            raise

        print(
            "[CACHE] retained_fragments="
            f"{global_index}; "
            "frame_grouping=g_ratio_only; "
            "excluded_segments="
            f"{skipped_excluded}; "
            "short_chunks="
            f"{skipped_short}"
        )

    def len(self):
        return len(self.files)

    def get_with_metadata(
        self,
        index,
    ):
        cache_object = torch.load(
            self.files[int(index)],
            map_location="cpu",
            weights_only=False,
        )
        graph_sequence = cache_object[
            "graph_seq"
        ]
        metadata = dict(
            cache_object.get(
                "metadata",
                {},
            )
            or {}
        )
        metadata["fragment_file"] = (
            self.files[int(index)].name
        )

        # Legacy caches produced by the original AutoLaparo script use
        # concept_name / target_label_name rather than counterpart_name.
        # Normalize those field names without modifying the cache files.
        raw_phase_id = int(
            metadata.get(
                "raw_phase_id",
                -1,
            )
        )

        if raw_phase_id not in COUNTERPART_SPECS:
            raise RuntimeError(
                "Cached fragment contains an unsupported "
                f"AutoLaparo raw phase: {raw_phase_id}"
            )

        counterpart_spec = (
            COUNTERPART_SPECS[
                raw_phase_id
            ]
        )
        expected_target_label = int(
            counterpart_spec[
                "target_label_id"
            ]
        )

        cached_target_label = int(
            metadata.get(
                "target_label_id",
                graph_sequence[0]
                .y.item(),
            )
        )

        if (
            cached_target_label
            != expected_target_label
        ):
            raise RuntimeError(
                "Cached target label does not match "
                "the fixed source-counterpart mapping "
                f"for raw phase {raw_phase_id}: "
                f"found {cached_target_label}, "
                f"expected {expected_target_label}."
            )

        metadata[
            "target_label_id"
        ] = cached_target_label

        metadata[
            "raw_phase_name"
        ] = str(
            metadata.get(
                "raw_phase_name",
                AUTOLAPARO_PHASE_NAMES[
                    raw_phase_id
                ],
            )
        )

        metadata[
            "counterpart_name"
        ] = str(
            metadata.get(
                "counterpart_name",
                metadata.get(
                    "target_label_name",
                    metadata.get(
                        "concept_name",
                        counterpart_spec[
                            "counterpart_name"
                        ],
                    ),
                ),
            )
        )

        # Keep the fixed study name authoritative even if an older cache
        # used a broader internal concept label such as "Dissection".
        metadata[
            "counterpart_name"
        ] = str(
            counterpart_spec[
                "counterpart_name"
            ]
        )

        return (
            graph_sequence,
            metadata,
        )

    def get(
        self,
        index,
    ):
        return self.get_with_metadata(
            index
        )[0]


In [ ]:
# =========================================================
# 5) FROZEN SOURCE-TRAINED MOTIF EXTRACTOR
# =========================================================

class PhaseInitializedSSTKGMotifExtractor(nn.Module):
    def __init__(
        self,
        in_channels=11,
        edge_dim=3,
        hidden_dim=128,
        motif_dim=128,
        num_phases=8,
        motif_window=16,
        motif_stride=8,
    ):
        super().__init__()

        self.hidden_dim = int(hidden_dim)
        self.motif_window = int(motif_window)
        self.motif_stride = int(motif_stride)

        self.graph_conv_1 = GATv2Conv(
            in_channels,
            hidden_dim,
            edge_dim=edge_dim,
        )
        self.graph_conv_2 = GATv2Conv(
            hidden_dim,
            hidden_dim,
            edge_dim=edge_dim,
        )
        self.virtual_context_gate = nn.Linear(
            hidden_dim * 2,
            1,
        )
        self.temporal_encoder = nn.GRU(
            hidden_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.phase_head = nn.Linear(
            hidden_dim * 3,
            num_phases,
        )
        self.window_fusion = nn.Linear(
            hidden_dim * 2,
            hidden_dim,
        )
        self.motif_projection = nn.Sequential(
            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_dim,
                motif_dim,
            ),
        )
        self.motif_phase_head = nn.Linear(
            motif_dim,
            num_phases,
        )

    def encode_frame_graphs(
        self,
        graph_sequence,
    ):
        frame_embeddings = []

        for graph in graph_sequence:
            node_embeddings = (
                self.graph_conv_1(
                    graph.x,
                    graph.edge_index,
                    graph.edge_attr,
                )
                .relu()
            )
            node_embeddings = (
                self.graph_conv_2(
                    node_embeddings,
                    graph.edge_index,
                    graph.edge_attr,
                )
                .relu()
            )

            mean_embedding = (
                node_embeddings.mean(
                    dim=0,
                    keepdim=True,
                )
            )
            virtual_embedding = node_embeddings[
                graph.is_v
            ]

            if virtual_embedding.size(0) == 0:
                virtual_embedding = mean_embedding
            else:
                virtual_embedding = (
                    virtual_embedding.mean(
                        dim=0,
                        keepdim=True,
                    )
                )

            gate_weight = torch.sigmoid(
                self.virtual_context_gate(
                    torch.cat(
                        [
                            mean_embedding,
                            virtual_embedding,
                        ],
                        dim=-1,
                    )
                )
            )
            frame_embeddings.append(
                gate_weight
                * virtual_embedding
                + (
                    1.0
                    - gate_weight
                )
                * mean_embedding
            )

        if not frame_embeddings:
            raise RuntimeError(
                "The graph sequence is empty."
            )

        return (
            torch.stack(frame_embeddings)
            .squeeze(1)
        )

    def contextualize_frames(
        self,
        frame_embeddings,
    ):
        recurrent_output, hidden_state = (
            self.temporal_encoder(
                frame_embeddings.unsqueeze(0)
            )
        )
        recurrent_output = (
            recurrent_output.squeeze(0)
        )
        temporal_context = (
            0.5
            * (
                recurrent_output[
                    :,
                    :self.hidden_dim,
                ]
                + recurrent_output[
                    :,
                    self.hidden_dim:,
                ]
            )
        )
        contextualized_frames = (
            0.5
            * frame_embeddings
            + 0.5
            * temporal_context
        )
        final_recurrent_state = torch.cat(
            [
                hidden_state[0],
                hidden_state[1],
            ],
            dim=-1,
        )
        temporal_mean_skip = (
            frame_embeddings.mean(
                dim=0,
                keepdim=True,
            )
        )
        sequence_readout = torch.cat(
            [
                final_recurrent_state,
                temporal_mean_skip,
            ],
            dim=-1,
        )

        return (
            contextualized_frames,
            sequence_readout,
        )

    def fuse_window(
        self,
        window_embeddings,
    ):
        return self.window_fusion(
            torch.cat(
                [
                    window_embeddings.mean(
                        dim=0,
                        keepdim=True,
                    ),
                    window_embeddings[
                        -1
                    ].unsqueeze(0),
                ],
                dim=-1,
            )
        )

    def extract_motif_signatures(
        self,
        contextualized_frames,
    ):
        sequence_length = int(
            contextualized_frames.shape[0]
        )

        if sequence_length < self.motif_window:
            fused_windows = self.fuse_window(
                contextualized_frames
            )
        else:
            fused_windows = torch.cat(
                [
                    self.fuse_window(
                        contextualized_frames[
                            start:
                            start
                            + self.motif_window
                        ]
                    )
                    for start in range(
                        0,
                        sequence_length
                        - self.motif_window
                        + 1,
                        self.motif_stride,
                    )
                ],
                dim=0,
            )

        return F.normalize(
            self.motif_projection(
                fused_windows
            ),
            p=2,
            dim=-1,
        )

    def forward(
        self,
        graph_sequence,
    ):
        frame_embeddings = (
            self.encode_frame_graphs(
                graph_sequence
            )
        )
        (
            contextualized_frames,
            sequence_readout,
        ) = self.contextualize_frames(
            frame_embeddings
        )
        motif_signatures = (
            self.extract_motif_signatures(
                contextualized_frames
            )
        )

        return {
            "motif_logits": (
                self.motif_phase_head(
                    motif_signatures
                )
            ),
            "motif_signatures": (
                motif_signatures
            ),
            "sequence_logits": (
                self.phase_head(
                    sequence_readout
                )
            ),
        }


def convert_development_checkpoint_keys(
    state_dict,
):
    prefix_map = {
        "conv1.": "graph_conv_1.",
        "conv2.": "graph_conv_2.",
        "context_attn.": (
            "virtual_context_gate."
        ),
        "rnn.": "temporal_encoder.",
        "fc.": "phase_head.",
        "window_fuse.": "window_fusion.",
        "motif_proj.": "motif_projection.",
        "motif_cls.": "motif_phase_head.",
    }
    converted = {}

    for key, value in state_dict.items():
        converted_key = key

        for (
            old_prefix,
            new_prefix,
        ) in prefix_map.items():
            if key.startswith(old_prefix):
                converted_key = (
                    new_prefix
                    + key[
                        len(old_prefix):
                    ]
                )
                break

        converted[
            converted_key
        ] = value

    return converted


def load_motif_extractor(
    checkpoint_path,
):
    checkpoint_path = Path(
        checkpoint_path
    )

    model = (
        PhaseInitializedSSTKGMotifExtractor(
            in_channels=(
                NODE_FEATURE_DIM
            ),
            edge_dim=EDGE_FEATURE_DIM,
            hidden_dim=HIDDEN_DIM,
            motif_dim=MOTIF_DIM,
            num_phases=(
                NUM_SOURCE_PHASES
            ),
            motif_window=MOTIF_WINDOW,
            motif_stride=MOTIF_STRIDE,
        )
        .to(DEVICE)
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )
    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )
    state_dict = (
        convert_development_checkpoint_keys(
            state_dict
        )
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )
    model.eval()

    return model

In [ ]:
# =========================================================
# 6) CALIBRATED HUB AND SOURCE-COUNTERPART PROJECTION
# =========================================================

def prototype_probabilities(
    query_embeddings,
    prototype_embeddings,
):
    similarities = (
        l2_normalize(
            query_embeddings
        )
        @ l2_normalize(
            prototype_embeddings
        ).T
    )

    return softmax_array(
        similarities,
        axis=1,
    )


def apply_dirichlet_calibrator(
    query_embeddings,
    prototype_embeddings,
    calibrator,
):
    raw_probabilities = np.clip(
        prototype_probabilities(
            query_embeddings,
            prototype_embeddings,
        ),
        EPS,
        1.0,
    )
    log_probabilities = np.log(
        raw_probabilities
    )
    coefficients = np.asarray(
        calibrator["coef"],
        dtype=np.float64,
    )
    intercept = np.asarray(
        calibrator["intercept"],
        dtype=np.float64,
    )
    classes = np.asarray(
        calibrator["classes"],
        dtype=int,
    )

    calibrated_logits = (
        log_probabilities
        @ coefficients.T
        + intercept.reshape(1, -1)
    )
    observed_probabilities = (
        softmax_array(
            calibrated_logits,
            axis=1,
        )
    )

    probabilities = np.zeros(
        (
            len(query_embeddings),
            NUM_SOURCE_PHASES,
        ),
        dtype=np.float64,
    )

    for (
        column_index,
        source_model_index,
    ) in enumerate(classes):
        probabilities[
            :,
            int(source_model_index),
        ] = observed_probabilities[
            :,
            column_index,
        ]

    return probabilities / (
        probabilities.sum(
            axis=1,
            keepdims=True,
        )
        + EPS
    )


def load_calibrated_hub(
    hub_path,
):
    hub_object = torch.load(
        Path(hub_path),
        map_location="cpu",
        weights_only=False,
    )

    prototype_embeddings = l2_normalize(
        np.asarray(
            hub_object["embeddings"],
            dtype=np.float32,
        )
    )
    records = pd.DataFrame(
        hub_object["records"]
    )

    if prototype_embeddings.shape != (
        NUM_SOURCE_PHASES,
        MOTIF_DIM,
    ):
        raise RuntimeError(
            "Unexpected hub prototype shape: "
            f"{prototype_embeddings.shape}"
        )

    # Preserve the saved prototype order used by the original experiment.
    # Validate it rather than silently sorting or reindexing it.
    if "verified_phase_id" in records.columns:
        saved_phase_ids = (
            records[
                "verified_phase_id"
            ]
            .astype(int)
            .tolist()
        )

        if saved_phase_ids != list(
            range(NUM_SOURCE_PHASES)
        ):
            raise RuntimeError(
                "Hub records are not stored in source "
                "model-index order 0..7. The notebook "
                "will not silently reorder prototypes."
            )

    calibration = dict(
        hub_object["calibration"]
    )
    dirichlet_calibrator = (
        calibration[
            "dirichlet_calibrator"
        ]
    )
    conformal_calibrator = (
        calibration[
            "conformal_calibrator"
        ]
    )

    calibrator_classes = np.asarray(
        dirichlet_calibrator[
            "classes"
        ],
        dtype=int,
    )

    if (
        calibrator_classes.min() < 0
        or calibrator_classes.max()
        >= NUM_SOURCE_PHASES
    ):
        raise RuntimeError(
            "Dirichlet calibrator classes must be "
            "0-based source model indices."
        )

    if "qhat" not in conformal_calibrator:
        raise RuntimeError(
            "The conformal threshold qhat "
            "is absent from the hub."
        )

    return {
        "prototype_embeddings": (
            prototype_embeddings
        ),
        "dirichlet_calibrator": (
            dirichlet_calibrator
        ),
        "conformal_calibrator": (
            conformal_calibrator
        ),
    }

def source_probabilities_to_counterparts(
    source_probabilities,
):
    source_probabilities = np.asarray(
        source_probabilities,
        dtype=np.float64,
    )
    counterpart_probabilities = np.zeros(
        len(TARGET_LABEL_TO_NAME),
        dtype=np.float64,
    )

    for (
        raw_phase_id,
        counterpart,
    ) in COUNTERPART_SPECS.items():
        target_label_id = int(
            counterpart[
                "target_label_id"
            ]
        )
        source_indices = np.asarray(
            counterpart[
                "source_model_indices"
            ],
            dtype=int,
        )
        counterpart_probabilities[
            target_label_id
        ] = float(
            source_probabilities[
                source_indices
            ].sum()
        )

    total = counterpart_probabilities.sum()

    if total <= EPS:
        counterpart_probabilities[:] = (
            1.0
            / len(
                counterpart_probabilities
            )
        )
    else:
        counterpart_probabilities /= total

    return counterpart_probabilities

In [ ]:
# =========================================================
# 7) UNCERTAINTY-SELECTED QUERY EXTRACTION
# =========================================================

def collect_uncertainty_selected_queries(
    model,
    dataset,
    hub,
):
    rows = []
    embeddings = []

    with torch.no_grad():
        for dataset_index in tqdm(
            range(len(dataset)),
            desc=(
                "Selecting maximum-entropy "
                "AutoLaparo motifs"
            ),
        ):
            (
                graph_sequence_cpu,
                metadata,
            ) = dataset.get_with_metadata(
                dataset_index
            )
            graph_sequence = [
                graph.to(DEVICE)
                for graph
                in graph_sequence_cpu
            ]
            model_output = model(
                graph_sequence
            )

            motif_probabilities = (
                softmax_array(
                    model_output[
                        "motif_logits"
                    ]
                    .detach()
                    .cpu()
                    .numpy(),
                    axis=1,
                )
            )
            motif_signatures = l2_normalize(
                model_output[
                    "motif_signatures"
                ]
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            motif_entropies = np.asarray(
                [
                    normalized_entropy(
                        probability
                    )
                    for probability
                    in motif_probabilities
                ],
                dtype=np.float64,
            )
            query_index = int(
                np.argmax(
                    motif_entropies
                )
            )
            query_embedding = (
                motif_signatures[
                    query_index
                ]
            )

            source_hub_probabilities = (
                apply_dirichlet_calibrator(
                    query_embedding.reshape(
                        1,
                        -1,
                    ),
                    hub[
                        "prototype_embeddings"
                    ],
                    hub[
                        "dirichlet_calibrator"
                    ],
                )[0]
            )
            counterpart_probabilities = (
                source_probabilities_to_counterparts(
                    source_hub_probabilities
                )
            )

            embedding_row = len(
                embeddings
            )
            embeddings.append(
                query_embedding
            )

            target_label_id = int(
                graph_sequence_cpu[0]
                .y.item()
            )

            rows.append(
                {
                    "embedding_row": int(
                        embedding_row
                    ),
                    "dataset_index": int(
                        dataset_index
                    ),
                    "case_id": str(
                        metadata[
                            "case_id"
                        ]
                    ),
                    "prefix": str(
                        metadata[
                            "prefix"
                        ]
                    ),
                    "fragment_file": str(
                        metadata[
                            "fragment_file"
                        ]
                    ),
                    "raw_phase_id": int(
                        metadata[
                            "raw_phase_id"
                        ]
                    ),
                    "raw_phase_name": str(
                        metadata[
                            "raw_phase_name"
                        ]
                    ),
                    "target_label_id": int(
                        target_label_id
                    ),
                    "counterpart_name": str(
                        metadata.get(
                            "counterpart_name",
                            COUNTERPART_SPECS[
                                int(
                                    metadata[
                                        "raw_phase_id"
                                    ]
                                )
                            ][
                                "counterpart_name"
                            ],
                        )
                    ),
                    "num_motifs": int(
                        len(motif_signatures)
                    ),
                    "selected_query_motif_index": int(
                        query_index
                    ),
                    "selected_query_entropy": float(
                        motif_entropies[
                            query_index
                        ]
                    ),
                    **{
                        (
                            "hub_counterpart_probability_"
                            f"{target_label_id}"
                        ): float(
                            counterpart_probabilities[
                                target_label_id
                            ]
                        )
                        for target_label_id
                        in range(
                            len(
                                TARGET_LABEL_TO_NAME
                            )
                        )
                    },
                }
            )

    query_table = pd.DataFrame(
        rows
    )
    embedding_matrix = l2_normalize(
        np.asarray(
            embeddings,
            dtype=np.float32,
        )
    )

    if (
        query_table[
            "case_id"
        ].nunique()
        != EXPECTED_REMOTE_CASES
    ):
        raise RuntimeError(
            "Every retained remote case must contribute "
            "at least one counterpart query."
        )

    if len(query_table) != int(
        EXPECTED_RETAINED_QUERIES
    ):
        raise RuntimeError(
            "The retained AutoLaparo query count "
            f"is {len(query_table)}, expected "
            f"{EXPECTED_RETAINED_QUERIES}. "
            "Check that cache fragmentation uses "
            "g_ratio only."
        )

    present_raw_phases = set(
        query_table[
            "raw_phase_id"
        ].astype(int)
    )
    expected_raw_phases = set(
        COUNTERPART_SPECS
    )

    if present_raw_phases != expected_raw_phases:
        raise RuntimeError(
            "The retained query table does not contain "
            "the complete fixed AutoLaparo mapping. "
            f"Found {sorted(present_raw_phases)}; "
            f"expected {sorted(expected_raw_phases)}."
        )

    return (
        query_table,
        embedding_matrix,
    )

In [ ]:
# =========================================================
# 8) OUTER LOCO EXTERNAL STRESS-TEST PROTOCOL
# =========================================================

def local_prototypes(
    training_table,
    embedding_matrix,
):
    prototypes = []
    labels = []

    for target_label_id in sorted(
        training_table[
            "target_label_id"
        ].astype(int).unique()
    ):
        indices = (
            training_table[
                training_table[
                    "target_label_id"
                ].astype(int)
                == int(target_label_id)
            ][
                "embedding_row"
            ]
            .to_numpy(dtype=int)
        )
        class_embeddings = l2_normalize(
            embedding_matrix[
                indices
            ]
        )
        prototypes.append(
            l2_normalize(
                class_embeddings.mean(
                    axis=0
                )
            )
        )
        labels.append(
            int(target_label_id)
        )

    if not prototypes:
        raise RuntimeError(
            "No local support prototypes could be built."
        )

    return (
        l2_normalize(
            np.asarray(
                prototypes,
                dtype=np.float32,
            )
        ),
        np.asarray(
            labels,
            dtype=int,
        ),
    )


def local_probabilities(
    query_table,
    embedding_matrix,
    prototype_embeddings,
    prototype_labels,
    scale,
):
    query_embeddings = l2_normalize(
        embedding_matrix[
            query_table[
                "embedding_row"
            ].to_numpy(dtype=int)
        ]
    )
    logits = (
        query_embeddings
        @ l2_normalize(
            prototype_embeddings
        ).T
        * float(scale)
    )
    present_probabilities = (
        softmax_array(
            logits,
            axis=1,
        )
    )
    output = np.zeros(
        (
            len(query_table),
            len(TARGET_LABEL_TO_NAME),
        ),
        dtype=np.float64,
    )

    for column_index, label in enumerate(
        prototype_labels
    ):
        output[
            :,
            int(label),
        ] = present_probabilities[
            :,
            column_index,
        ]

    totals = output.sum(
        axis=1,
        keepdims=True,
    )
    zero_rows = (
        totals[:, 0]
        <= EPS
    )
    output /= totals + EPS

    if np.any(zero_rows):
        output[
            zero_rows,
            :,
        ] = (
            1.0
            / output.shape[1]
        )

    return output


def hub_probabilities(
    query_table,
):
    columns = [
        (
            "hub_counterpart_probability_"
            f"{target_label_id}"
        )
        for target_label_id
        in range(
            len(
                TARGET_LABEL_TO_NAME
            )
        )
    ]
    probabilities = query_table[
        columns
    ].to_numpy(
        dtype=np.float64
    )

    return probabilities / (
        probabilities.sum(
            axis=1,
            keepdims=True,
        )
        + EPS
    )


def fuse_global(
    local_probability,
    hub_probability,
    alpha,
):
    fused = (
        (
            1.0
            - float(alpha)
        )
        * local_probability
        + float(alpha)
        * hub_probability
    )

    return fused / (
        fused.sum(
            axis=1,
            keepdims=True,
        )
        + EPS
    )


def fuse_per_counterpart(
    local_probability,
    hub_probability,
    alpha_vector,
):
    alpha_vector = np.asarray(
        alpha_vector,
        dtype=np.float64,
    ).reshape(1, -1)
    fused = (
        (
            1.0
            - alpha_vector
        )
        * local_probability
        + alpha_vector
        * hub_probability
    )

    return fused / (
        fused.sum(
            axis=1,
            keepdims=True,
        )
        + EPS
    )


def support_inner_probabilities(
    support_table,
    embedding_matrix,
    scale,
):
    true_labels = []
    local_rows = []
    hub_rows = []

    support_cases = sorted(
        support_table[
            "case_id"
        ].astype(str).unique()
    )

    for validation_case in support_cases:
        inner_training = support_table[
            support_table[
                "case_id"
            ].astype(str)
            != validation_case
        ].copy()
        inner_validation = support_table[
            support_table[
                "case_id"
            ].astype(str)
            == validation_case
        ].copy()

        if (
            len(inner_training) == 0
            or len(inner_validation) == 0
        ):
            continue

        (
            prototype_embeddings,
            prototype_labels,
        ) = local_prototypes(
            inner_training,
            embedding_matrix,
        )
        local_rows.append(
            local_probabilities(
                inner_validation,
                embedding_matrix,
                prototype_embeddings,
                prototype_labels,
                scale,
            )
        )
        hub_rows.append(
            hub_probabilities(
                inner_validation
            )
        )
        true_labels.extend(
            inner_validation[
                "target_label_id"
            ]
            .astype(int)
            .tolist()
        )

    if not true_labels:
        return (
            np.asarray(
                [],
                dtype=int,
            ),
            np.zeros(
                (
                    0,
                    len(
                        TARGET_LABEL_TO_NAME
                    ),
                ),
                dtype=np.float64,
            ),
            np.zeros(
                (
                    0,
                    len(
                        TARGET_LABEL_TO_NAME
                    ),
                ),
                dtype=np.float64,
            ),
        )

    return (
        np.asarray(
            true_labels,
            dtype=int,
        ),
        np.vstack(
            local_rows
        ),
        np.vstack(
            hub_rows
        ),
    )


def accuracy_from_probabilities(
    true_labels,
    probabilities,
):
    true_labels = np.asarray(
        true_labels,
        dtype=int,
    )

    if len(true_labels) == 0:
        return np.nan

    return float(
        np.mean(
            np.argmax(
                probabilities,
                axis=1,
            )
            == true_labels
        )
    )


def select_local_scale(
    support_table,
    embedding_matrix,
):
    best_scale = (
        LOCAL_SCALE_GRID[0]
    )
    best_accuracy = float(
        "-inf"
    )

    for scale in LOCAL_SCALE_GRID:
        (
            true_labels,
            local_probability,
            _,
        ) = support_inner_probabilities(
            support_table,
            embedding_matrix,
            scale,
        )
        accuracy = (
            accuracy_from_probabilities(
                true_labels,
                local_probability,
            )
            if len(true_labels)
            else float("-inf")
        )

        if accuracy > best_accuracy:
            best_accuracy = float(
                accuracy
            )
            best_scale = float(
                scale
            )

    return best_scale


def select_global_alpha(
    true_labels,
    local_probability,
    hub_probability,
):
    if len(true_labels) == 0:
        return 0.0

    best_alpha = 0.0
    best_accuracy = float(
        "-inf"
    )

    for alpha in ALPHA_GRID:
        accuracy = (
            accuracy_from_probabilities(
                true_labels,
                fuse_global(
                    local_probability,
                    hub_probability,
                    alpha,
                ),
            )
        )

        if accuracy > best_accuracy:
            best_accuracy = float(
                accuracy
            )
            best_alpha = float(
                alpha
            )

    return best_alpha


def select_per_counterpart_alpha(
    true_labels,
    local_probability,
    hub_probability,
):
    if len(true_labels) == 0:
        return np.zeros(
            len(TARGET_LABEL_TO_NAME),
            dtype=np.float64,
        )

    number_of_classes = len(
        TARGET_LABEL_TO_NAME
    )
    alpha_vector = np.zeros(
        number_of_classes,
        dtype=np.float64,
    )

    for target_label_id in range(
        number_of_classes
    ):
        mask = (
            np.asarray(
                true_labels,
                dtype=int,
            )
            == target_label_id
        )

        if not np.any(mask):
            continue

        best_alpha = 0.0
        best_accuracy = float(
            "-inf"
        )

        for alpha in ALPHA_GRID:
            # Exact original protocol: while selecting the
            # alpha for one counterpart, all other counterpart
            # weights are held at zero.
            candidate_vector = np.zeros(
                number_of_classes,
                dtype=np.float64,
            )
            candidate_vector[
                target_label_id
            ] = float(alpha)

            accuracy = (
                accuracy_from_probabilities(
                    np.asarray(
                        true_labels,
                        dtype=int,
                    )[mask],
                    fuse_per_counterpart(
                        local_probability[
                            mask
                        ],
                        hub_probability[
                            mask
                        ],
                        candidate_vector,
                    ),
                )
            )

            if accuracy > best_accuracy:
                best_accuracy = float(
                    accuracy
                )
                best_alpha = float(
                    alpha
                )

        alpha_vector[
            target_label_id
        ] = best_alpha

    return alpha_vector

def fit_support_reliability(
    true_labels,
    candidate_probabilities,
):
    true_labels = np.asarray(
        true_labels,
        dtype=int,
    )
    rows = []

    for (
        method_name,
        probabilities,
    ) in candidate_probabilities.items():
        predictions = np.argmax(
            probabilities,
            axis=1,
        )
        correctness = (
            predictions
            == true_labels
        ).astype(float)
        overall_accuracy = float(
            correctness.mean()
        )

        for target_label_id in range(
            len(TARGET_LABEL_TO_NAME)
        ):
            mask = (
                predictions
                == target_label_id
            )
            support_count = int(
                mask.sum()
            )
            support_correct = int(
                correctness[
                    mask
                ].sum()
            ) if support_count else 0
            smoothed_reliability = (
                support_correct
                + RELIABILITY_PRIOR_STRENGTH
                * overall_accuracy
            ) / (
                support_count
                + RELIABILITY_PRIOR_STRENGTH
            )

            rows.append(
                {
                    "method": str(
                        method_name
                    ),
                    "predicted_target_label_id": int(
                        target_label_id
                    ),
                    "smoothed_reliability": float(
                        smoothed_reliability
                    ),
                }
            )

    return pd.DataFrame(
        rows
    )


def apply_support_reliability_gate(
    candidate_probabilities,
    reliability_table,
):
    reliability_lookup = {
        (
            str(row["method"]),
            int(
                row[
                    "predicted_target_label_id"
                ]
            ),
        ): float(
            row[
                "smoothed_reliability"
            ]
        )
        for _, row
        in reliability_table.iterrows()
    }

    number_of_rows = len(
        next(
            iter(
                candidate_probabilities.values()
            )
        )
    )
    final_probabilities = np.zeros(
        (
            number_of_rows,
            len(TARGET_LABEL_TO_NAME),
        ),
        dtype=np.float64,
    )
    selected_methods = []

    for row_index in range(
        number_of_rows
    ):
        best_method = None
        best_score = float(
            "-inf"
        )

        for (
            method_name,
            probabilities,
        ) in candidate_probabilities.items():
            probability = probabilities[
                row_index
            ]
            prediction = int(
                np.argmax(
                    probability
                )
            )
            confidence = float(
                np.max(
                    probability
                )
            )
            score = (
                reliability_lookup.get(
                    (
                        method_name,
                        prediction,
                    ),
                    0.5,
                )
                + 1e-4
                * confidence
            )

            if score > best_score:
                best_score = score
                best_method = method_name

        final_probabilities[
            row_index
        ] = candidate_probabilities[
            best_method
        ][
            row_index
        ]
        selected_methods.append(
            str(best_method)
        )

    return (
        final_probabilities,
        selected_methods,
    )


def run_external_loco(
    query_table,
    embedding_matrix,
):
    prediction_rows = []
    case_ids = sorted(
        query_table[
            "case_id"
        ].astype(str).unique()
    )

    for fold_id, held_out_case in enumerate(
        case_ids
    ):
        support_table = query_table[
            query_table[
                "case_id"
            ].astype(str)
            != held_out_case
        ].copy()
        held_out_table = query_table[
            query_table[
                "case_id"
            ].astype(str)
            == held_out_case
        ].copy().reset_index(
            drop=True
        )

        selected_scale = (
            select_local_scale(
                support_table,
                embedding_matrix,
            )
        )
        (
            inner_true,
            inner_local,
            inner_hub,
        ) = support_inner_probabilities(
            support_table,
            embedding_matrix,
            selected_scale,
        )
        selected_global_alpha = (
            select_global_alpha(
                inner_true,
                inner_local,
                inner_hub,
            )
        )
        selected_per_counterpart_alpha = (
            select_per_counterpart_alpha(
                inner_true,
                inner_local,
                inner_hub,
            )
        )

        inner_global = fuse_global(
            inner_local,
            inner_hub,
            selected_global_alpha,
        )
        inner_per_counterpart = (
            fuse_per_counterpart(
                inner_local,
                inner_hub,
                selected_per_counterpart_alpha,
            )
        )
        reliability_table = (
            fit_support_reliability(
                inner_true,
                {
                    "local": inner_local,
                    "hub": inner_hub,
                    "global_fusion": (
                        inner_global
                    ),
                    "per_counterpart_fusion": (
                        inner_per_counterpart
                    ),
                },
            )
        )

        (
            prototype_embeddings,
            prototype_labels,
        ) = local_prototypes(
            support_table,
            embedding_matrix,
        )
        local_probability = (
            local_probabilities(
                held_out_table,
                embedding_matrix,
                prototype_embeddings,
                prototype_labels,
                selected_scale,
            )
        )
        hub_probability = (
            hub_probabilities(
                held_out_table
            )
        )
        global_probability = (
            fuse_global(
                local_probability,
                hub_probability,
                selected_global_alpha,
            )
        )
        per_counterpart_probability = (
            fuse_per_counterpart(
                local_probability,
                hub_probability,
                selected_per_counterpart_alpha,
            )
        )
        (
            final_probability,
            selected_methods,
        ) = apply_support_reliability_gate(
            {
                "local": (
                    local_probability
                ),
                "hub": hub_probability,
                "global_fusion": (
                    global_probability
                ),
                "per_counterpart_fusion": (
                    per_counterpart_probability
                ),
            },
            reliability_table,
        )

        local_predictions = np.argmax(
            local_probability,
            axis=1,
        )
        hub_predictions = np.argmax(
            hub_probability,
            axis=1,
        )
        final_predictions = np.argmax(
            final_probability,
            axis=1,
        )

        for row_index, (
            _,
            query_row,
        ) in enumerate(
            held_out_table.iterrows()
        ):
            true_target = int(
                query_row[
                    "target_label_id"
                ]
            )

            prediction_rows.append(
                {
                    "fold_id": int(
                        fold_id
                    ),
                    "held_out_case": str(
                        held_out_case
                    ),
                    "dataset_index": int(
                        query_row[
                            "dataset_index"
                        ]
                    ),
                    "case_id": str(
                        query_row[
                            "case_id"
                        ]
                    ),
                    "prefix": str(
                        query_row[
                            "prefix"
                        ]
                    ),
                    "fragment_file": str(
                        query_row[
                            "fragment_file"
                        ]
                    ),
                    "raw_phase_id": int(
                        query_row[
                            "raw_phase_id"
                        ]
                    ),
                    "raw_phase_name": str(
                        query_row[
                            "raw_phase_name"
                        ]
                    ),
                    "true_target_label_id": int(
                        true_target
                    ),
                    "true_counterpart_name": (
                        target_name(
                            true_target
                        )
                    ),
                    "selected_query_motif_index": int(
                        query_row[
                            "selected_query_motif_index"
                        ]
                    ),
                    "selected_query_entropy": float(
                        query_row[
                            "selected_query_entropy"
                        ]
                    ),
                    "selected_local_scale": float(
                        selected_scale
                    ),
                    "selected_global_alpha": float(
                        selected_global_alpha
                    ),
                    "selected_per_counterpart_alpha": json.dumps(
                        [
                            float(value)
                            for value
                            in selected_per_counterpart_alpha
                        ]
                    ),
                    "target_support_prediction": int(
                        local_predictions[
                            row_index
                        ]
                    ),
                    "target_support_prediction_name": (
                        target_name(
                            local_predictions[
                                row_index
                            ]
                        )
                    ),
                    "target_support_correct": int(
                        local_predictions[
                            row_index
                        ]
                        == true_target
                    ),
                    "hub_only_prediction": int(
                        hub_predictions[
                            row_index
                        ]
                    ),
                    "hub_only_prediction_name": (
                        target_name(
                            hub_predictions[
                                row_index
                            ]
                        )
                    ),
                    "hub_only_correct": int(
                        hub_predictions[
                            row_index
                        ]
                        == true_target
                    ),
                    "final_dmet_prediction": int(
                        final_predictions[
                            row_index
                        ]
                    ),
                    "final_dmet_prediction_name": (
                        target_name(
                            final_predictions[
                                row_index
                            ]
                        )
                    ),
                    "final_dmet_correct": int(
                        final_predictions[
                            row_index
                        ]
                        == true_target
                    ),
                    "selected_candidate_method": str(
                        selected_methods[
                            row_index
                        ]
                    ),
                }
            )

    return pd.DataFrame(
        prediction_rows
    )

In [ ]:
# =========================================================
# 9) MANUSCRIPT SUMMARY AND THREE-FILE OUTPUT
# =========================================================

def build_external_summary(
    predictions,
):
    target_support_accuracy = (
        100.0
        * float(
            predictions[
                "target_support_correct"
            ].mean()
        )
    )
    hub_only_accuracy = (
        100.0
        * float(
            predictions[
                "hub_only_correct"
            ].mean()
        )
    )
    final_accuracy = (
        100.0
        * float(
            predictions[
                "final_dmet_correct"
            ].mean()
        )
    )

    case_gains = (
        predictions.assign(
            gain_in_correctness=(
                predictions[
                    "final_dmet_correct"
                ].astype(int)
                - predictions[
                    "target_support_correct"
                ].astype(int)
            )
        )
        .groupby(
            "case_id"
        )[
            "gain_in_correctness"
        ]
        .mean()
        .to_numpy(
            dtype=float
        )
        * 100.0
    )

    return pd.DataFrame(
        [
            {
                "dataset": "AutoLaparo",
                "remote_cases": int(
                    predictions[
                        "case_id"
                    ].nunique()
                ),
                "retained_queries": int(
                    len(predictions)
                ),
                "target_support_baseline_accuracy_percent": float(
                    target_support_accuracy
                ),
                "hub_only_consultation_accuracy_percent": float(
                    hub_only_accuracy
                ),
                "final_dmet_sstkg_accuracy_percent": float(
                    final_accuracy
                ),
                "micro_gain_vs_baseline_pp": float(
                    final_accuracy
                    - target_support_accuracy
                ),
                "mean_case_level_gain_vs_baseline_pp": float(
                    case_gains.mean()
                ),
            }
        ]
    )


def save_external_results(
    run_directory,
    summary,
    predictions,
    config,
):
    run_directory = Path(
        run_directory
    )
    run_directory.mkdir(
        parents=True,
        exist_ok=False,
    )

    output_paths = {
        "summary": (
            run_directory
            / (
                "AutoLaparo_"
                "ExternalStressTestSummary.csv"
            )
        ),
        "predictions": (
            run_directory
            / (
                "AutoLaparo_"
                "ExternalStressTestPredictions.csv"
            )
        ),
        "config": (
            run_directory
            / (
                "AutoLaparo_"
                "ExternalStressTestConfig.json"
            )
        ),
    }

    summary.to_csv(
        output_paths["summary"],
        index=False,
    )
    predictions.to_csv(
        output_paths[
            "predictions"
        ],
        index=False,
    )
    output_paths[
        "config"
    ].write_text(
        json.dumps(
            config,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    missing_outputs = [
        path
        for path in output_paths.values()
        if not path.is_file()
    ]

    if missing_outputs:
        raise RuntimeError(
            "Expected external-stress-test "
            f"outputs were not generated: "
            f"{missing_outputs}"
        )

    return output_paths

In [ ]:
# =========================================================
# 10) COMPLETE AUTOLAPARO PIPELINE
# =========================================================

def run_autolaparo_external_stress_test():
    set_seed(SEED)
    validate_input_paths()

    results_root = Path(
        RESULTS_DIR
    ).expanduser().resolve()
    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )
    run_directory = (
        results_root
        / (
            "autolaparo_external_stress_test_"
            f"{timestamp}"
        )
    )

    print(
        "\n"
        + "=" * 88
    )
    print(
        "AUTOLAPARO EXTERNAL "
        "SOURCE-COUNTERPART STRESS TEST"
    )
    print(
        "=" * 88
    )
    print(
        f"Remote S-STKG root: "
        f"{REMOTE_SSTKG_ROOT}"
    )
    print(
        f"Managed cache: "
        f"{REMOTE_CACHE_DIR}"
    )
    print(
        f"Motif checkpoint: "
        f"{MOTIF_CHECKPOINT}"
    )
    print(
        f"Calibrated hub: "
        f"{HUB_FILE}"
    )
    print(
        f"Run directory: "
        f"{run_directory}"
    )

    dataset = (
        AutoLaparoCounterpartDataset(
            root_dir=(
                REMOTE_SSTKG_ROOT
            ),
            cache_dir=(
                REMOTE_CACHE_DIR
            ),
            chunk_size=CHUNK_SIZE,
            minimum_chunk_frames=(
                MIN_CHUNK_FRAMES
            ),
            force_reprocess=(
                FORCE_REPROCESS_CACHE
            ),
        )
    )
    model = load_motif_extractor(
        MOTIF_CHECKPOINT
    )
    hub = load_calibrated_hub(
        HUB_FILE
    )
    (
        query_table,
        embedding_matrix,
    ) = collect_uncertainty_selected_queries(
        model,
        dataset,
        hub,
    )
    predictions = run_external_loco(
        query_table,
        embedding_matrix,
    )
    summary = build_external_summary(
        predictions
    )

    config = {
        "experiment": (
            "autolaparo_external_"
            "source_counterpart_stress_test"
        ),
        "inputs": {
            "remote_sstkg_root": str(
                Path(
                    REMOTE_SSTKG_ROOT
                ).expanduser().resolve()
            ),
            "remote_cache_dir": str(
                Path(
                    REMOTE_CACHE_DIR
                ).expanduser().resolve()
            ),
            "motif_checkpoint": str(
                Path(
                    MOTIF_CHECKPOINT
                ).expanduser().resolve()
            ),
            "hub_file": str(
                Path(
                    HUB_FILE
                ).expanduser().resolve()
            ),
        },
        "fixed_counterpart_mapping": {
            str(raw_phase_id): {
                "autolaparo_phase_name": (
                    AUTOLAPARO_PHASE_NAMES[
                        raw_phase_id
                    ]
                ),
                "target_label_id": int(
                    spec[
                        "target_label_id"
                    ]
                ),
                "counterpart_name": str(
                    spec[
                        "counterpart_name"
                    ]
                ),
                "source_original_phase_ids": [
                    int(value)
                    for value
                    in spec[
                        "source_original_phase_ids"
                    ]
                ],
                "source_phase_names": [
                    str(value)
                    for value
                    in spec[
                        "source_phase_names"
                    ]
                ],
            }
            for raw_phase_id, spec
            in COUNTERPART_SPECS.items()
        },
        "excluded_autolaparo_raw_phases": {
            str(raw_phase_id): str(
                phase_name
            )
            for raw_phase_id, phase_name
            in AUTOLAPARO_PHASE_NAMES.items()
            if raw_phase_id
            not in COUNTERPART_SPECS
        },
        "protocol": {
            "remote_cases": int(
                EXPECTED_REMOTE_CASES
            ),
            "expected_retained_queries": int(
                EXPECTED_RETAINED_QUERIES
            ),
            "cache_fragmentation": (
                "unique g_ratio values only; p_ratio "
                "is not used to split sampled frames"
            ),
            "query_selection": (
                "one maximum-entropy motif per "
                "retained phase-aligned fragment "
                "in the original eight-phase source space"
            ),
            "outer_evaluation": (
                "leave-one-case-out"
            ),
            "target_support_baseline": (
                "normalized local prototypes built "
                "from the non-held-out support cases"
            ),
            "hub_only": (
                "Dirichlet-calibrated source-hub "
                "distribution projected into the fixed "
                "counterpart concept space"
            ),
            "final_method": (
                "support-calibrated reliability gate "
                "over local, hub, global-fusion, and "
                "per-counterpart-fusion candidates"
            ),
            "parameter_selection": (
                "inner case-wise validation using only "
                "the support cases of each outer fold"
            ),
            "local_scale_grid": [
                float(value)
                for value
                in LOCAL_SCALE_GRID
            ],
            "fusion_alpha_grid": [
                float(value)
                for value
                in ALPHA_GRID
            ],
            "reliability_prior_strength": float(
                RELIABILITY_PRIOR_STRENGTH
            ),
            "seed": int(SEED),
        },
        "communication": {
            "query_bytes": int(
                QUERY_BYTES
            ),
            "response_bytes": int(
                RESPONSE_BYTES
            ),
            "bytes_per_consultation": int(
                BYTES_PER_CONSULTATION
            ),
            "logical_payload_for_retained_queries_mib": float(
                logical_payload_mib(
                    len(predictions)
                )
            ),
            "hub_artifact_excluded_from_query_response_payload": True,
            "transport_and_system_overhead_excluded": True,
        },
        "manuscript_reference": {
            key: (
                int(value)
                if key == "retained_queries"
                else float(value)
            )
            for key, value
            in MANUSCRIPT_REFERENCE.items()
        },
        "generated_files": {
            "summary_csv": (
                "manuscript Table 7 metrics"
            ),
            "predictions_csv": (
                "held-out local, hub-only, and final "
                "query-level predictions"
            ),
            "config_json": (
                "fixed mapping, inputs, and support-only "
                "selection protocol"
            ),
        },
    }

    output_paths = save_external_results(
        run_directory,
        summary,
        predictions,
        config,
    )

    print(
        "\n[EXTERNAL STRESS-TEST SUMMARY]"
    )
    print(
        summary.to_string(
            index=False
        )
    )
    reference_rows = []
    computed_row = summary.iloc[0]

    for metric_name, reference_value in (
        MANUSCRIPT_REFERENCE.items()
    ):
        computed_value = float(
            computed_row[
                metric_name
            ]
        )
        reference_rows.append(
            {
                "metric": metric_name,
                "computed": computed_value,
                "manuscript": float(
                    reference_value
                ),
                "absolute_difference": float(
                    abs(
                        computed_value
                        - float(
                            reference_value
                        )
                    )
                ),
            }
        )

    manuscript_consistency = pd.DataFrame(
        reference_rows
    )

    print(
        "\n[MANUSCRIPT CONSISTENCY CHECK]"
    )
    print(
        manuscript_consistency.to_string(
            index=False
        )
    )

    print(
        "\n[GENERATED RESULT FILES]"
    )
    for name, path in (
        output_paths.items()
    ):
        print(
            f"  {name}: {path}"
        )

    return {
        "dataset": dataset,
        "model": model,
        "hub": hub,
        "query_table": query_table,
        "predictions": predictions,
        "summary": summary,
        "manuscript_consistency": (
            manuscript_consistency
        ),
        "output_paths": output_paths,
        "run_directory": run_directory,
    }

In [ ]:
external_run = (
    run_autolaparo_external_stress_test()
)

external_run["summary"]
